In [1]:
import pandas as pd
import numpy as np
import sqlite3
import csv


In [2]:
df_mit = pd.read_csv("/workspaces/Trabajo-Final-Analisis-de-Datos/MitM Dataset.csv")
#print(df_mit.head(2))
df_ddos_1 = pd.read_csv ("/workspaces/Trabajo-Final-Analisis-de-Datos/DDOS Dataset.csv")
#print(df_ddos.head(2))
df_fry = pd.read_csv ("/workspaces/Trabajo-Final-Analisis-de-Datos/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
df_ddos_2 = pd.read_csv("/workspaces/Trabajo-Final-Analisis-de-Datos/DDoSdata.csv")


/tmp/ipykernel_3868/968960442.py:6: DtypeWarning: Columns (0: daddr, 1: pkts) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ddos_2 = pd.read_csv("/workspaces/Trabajo-Final-Analisis-de-Datos/DDoSdata.csv")


In [5]:
"""RENOMBRANDO COLUMNAS"""
df_ddos_2 = df_ddos_2.rename(columns={
    'pkSeqID': 'id',
    'stime': 'start_time',
    'flgs': 'flags',
    'flgs_number': 'flags_number',
    'proto': 'protocol',
    'proto_number': 'protocol_number',
    'saddr': 'source_address',
    'sport': 'source_port',
    'daddr': 'destination_address',
    'dport': 'destination_port',
    'pkts': 'flow_packets',
    'Itime': 'initial_time',
    'dur': 'duration',
    'Itime' : 'initial_time',
    #####################################################################
    'spkts' : 'source_packets',
    'dpkts' : 'destination_packets',
    #relacion entre source_packets y destination_packets es importante
    'sbytes' : 'source_bytes',
    'dbytes' : 'destination_bytes',
    #esta relacion es como la anterior, la alta frecuencia incial seguida por una baja secuencia
    #de respuesta indica un flujo anomalo de paquetes
    #'Rate' : cantidad de trafico / tiempo
    'srate' : 'source_rate',
    #la presencia de muchas fuentes con tasas elevadas puede ser una señal muy interesante.
    'drate' : 'destination_rate',
    #La diferencia entre ambas puede ser significativa.
    'N_IN_Conn_P_DstIP' : 'num_incoming.connection_per_destip',
    #número de conexiones entrantes asociadas a una IP de destino.
    #los ataques ddos se caracterizan por atacar una misma victina desde multiples IP
    'N_IN_Conn_P_SrcIP' : 'num_incoming.connection_per_sourceip',
    #una misma IP participando en muchas conexiones puede ser relevante.
    'TnP_PSrcIP' : 'total_packets_from_sourceip',
    #representa el total de paquetes asociado a una IP de origen y cuanto trafico genera una ip
    'TnP_PDstIP' : 'total_packets_drom_destip'
    #lo mismo pero al inverso
})
df_ddos_2.columns

Index(['id', 'start_time', 'flags', 'flags_number', 'protocol',
       'protocol_number', 'source_address', 'source_port',
       'destination_address', 'destination_port', 'flow_packets', 'bytes',
       'state', 'state_number', 'ltime', 'seq', 'duration', 'mean', 'stddev',
       'sum', 'min', 'max', 'source_packets', 'destination_packets',
       'source_bytes', 'destination_bytes', 'rate', 'source_rate',
       'destination_rate', 'TnBPSrcIP', 'TnBPDstIP',
       'total_packets_from_sourceip', 'total_packets_drom_destip',
       'TnP_PerProto', 'TnP_Per_Dport', 'AR_P_Proto_P_SrcIP',
       'AR_P_Proto_P_DstIP', 'num_incoming.connection_per_destip',
       'num_incoming.connection_per_sourceip', 'AR_P_Proto_P_Sport',
       'AR_P_Proto_P_Dport', 'Pkts_P_State_P_Protocol_P_DestIP',
       'Pkts_P_State_P_Protocol_P_SrcIP', 'attack', 'category', 'subcategory'],
      dtype='str')

MUCHOS PAQUETES
              +
TASA MUY ELEVADA
              +
MUCHAS CONEXIONES
              +
MUCHAS IPs origen
              +
POCA RESPUESTA
 = DDoS

In [15]:
"""Limpieza dataset ddos_2 basado en el video de
https://www.youtube.com/watch?v=QqEdSk98Wqs KNOWLEDGE DOCTOR"""
df_ddos_2.info()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
display(df_ddos_2.head(20))


<class 'pandas.DataFrame'>
Index: 1927101 entries, 1650261 to 3577363
Data columns (total 46 columns):
 #   Column                                Dtype  
---  ------                                -----  
 0   id                                    int64  
 1   start_time                            float64
 2   flags                                 str    
 3   flags_number                          int64  
 4   protocol                              str    
 5   protocol_number                       int64  
 6   source_address                        str    
 7   source_port                           object 
 8   destination_address                   str    
 9   destination_port                      object 
 10  flow_packets                          int64  
 11  bytes                                 int64  
 12  state                                 str    
 13  state_number                          int64  
 14  ltime                                 float64
 15  seq                      

,id,start_time,flags,flags_number,protocol,protocol_number,source_address,source_port,destination_address,destination_port,flow_packets,bytes,state,state_number,ltime,seq,duration,mean,stddev,sum,min,max,source_packets,destination_packets,source_bytes,destination_bytes,rate,source_rate,destination_rate,TnBPSrcIP,TnBPDstIP,total_packets_from_sourceip,total_packets_drom_destip,TnP_PerProto,TnP_Per_Dport,AR_P_Proto_P_SrcIP,AR_P_Proto_P_DstIP,num_incoming.connection_per_destip,num_incoming.connection_per_sourceip,AR_P_Proto_P_Sport,AR_P_Proto_P_Dport,Pkts_P_State_P_Protocol_P_DestIP,Pkts_P_State_P_Protocol_P_SrcIP,attack,category,subcategory
1650261,1650261,1.528103e+09,e,1,tcp,1,192.168.100.150,54110,192.168.100.3,80,10,1729,RST,1,1.528103e+09,20,6.406424,0.679473,0.544126,1.358946,0.135347,1.223599,6,4,963,766,1.404840,0.780467,0.468280,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.56093,1.21662,328,308,1,DDoS,HTTP
1650262,1650262,1.528103e+09,e,1,tcp,1,192.168.100.150,54112,192.168.100.3,80,10,1604,RST,1,1.528103e+09,21,6.405851,0.679572,0.544197,1.359144,0.135375,1.223769,6,4,838,766,1.404966,0.780536,0.468322,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.56107,1.21662,328,308,1,DDoS,HTTP
1650263,1650263,1.528103e+09,e,1,tcp,1,192.168.100.150,54114,192.168.100.3,80,8,1708,RST,1,1.528103e+09,22,6.401038,1.110847,1.110847,2.221694,0.000000,2.221694,5,3,1008,700,1.093573,0.624899,0.900214,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.24980,1.21662,328,308,1,DDoS,HTTP
1650264,1650264,1.528103e+09,e,1,tcp,1,192.168.100.150,54116,192.168.100.3,80,8,1462,RST,1,1.528103e+09,23,6.400703,1.113328,1.113328,2.226655,0.000000,2.226655,5,3,762,700,1.093630,0.624931,0.898208,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.24986,1.21662,328,308,1,DDoS,HTTP
1650265,1650265,1.528103e+09,e,1,tcp,1,192.168.100.150,54118,192.168.100.3,80,8,1296,RST,1,1.528103e+09,24,6.400472,1.113098,1.113098,2.226195,0.000000,2.226195,5,3,596,700,1.093669,0.624954,0.898394,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.24991,1.21662,328,308,1,DDoS,HTTP
1650266,1650266,1.528103e+09,e,1,tcp,1,192.168.100.150,54120,192.168.100.3,80,8,1434,RST,1,1.528103e+09,25,6.400154,1.113392,1.113392,2.226783,0.000000,2.226783,5,3,734,700,1.093724,0.624985,0.898157,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.24997,1.21662,328,308,1,DDoS,HTTP
1650267,1650267,1.528103e+09,e,1,tcp,1,192.168.100.150,54122,192.168.100.3,80,8,1764,RST,1,1.528103e+09,26,6.399644,1.612942,1.612942,3.225885,0.000000,3.225885,5,3,1064,700,1.093811,0.625035,0.619985,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.25007,1.21662,328,308,1,DDoS,HTTP
1650268,1650268,1.528103e+09,e,1,tcp,1,192.168.100.150,54124,192.168.100.3,80,8,1469,RST,1,1.528103e+09,27,6.399428,1.613153,1.613153,3.226305,0.000000,3.226305,5,3,769,700,1.093848,0.625056,0.619904,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.25011,1.21662,328,308,1,DDoS,HTTP
1650269,1650269,1.528103e+09,e,1,tcp,1,192.168.100.150,54126,192.168.100.3,80,8,1613,RST,1,1.528103e+09,28,6.396386,1.611953,1.611953,3.223905,0.000000,3.223905,5,3,913,700,1.094368,0.625353,0.620366,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.25071,1.21662,328,308,1,DDoS,HTTP
1650270,1650270,1.528103e+09,e,1,tcp,1,192.168.100.150,54128,192.168.100.3,80,8,1376,RST,1,1.528103e+09,29,6.396170,1.612818,1.612818,3.225637,0.000000,3.225637,5,3,676,700,1.094405,0.625374,0.620033,56864,59969,308,328,328,700,1.26889,1.21662,40,38,1.25075,1.21662,328,308,1,DDoS,HTTP
